In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pgmpy.inference import VariableElimination
from pgmpy.readwrite import XMLBIFReader
import random

from pgmpy.readwrite import XMLBIFReader
import itertools

import pysmile
import pysmile_license
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import pdb
import os

from v2.costs_and_utilities import *
from v2.patients import patient
from v2.dist_prob_cit import plot_histograms_count_distrib
from v2.get_combinations import *

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel


In [8]:
# ---------------------- Run the Simulation ----------------------
limit = True

total_sim = 100

n_K_points = 20
upper_K = 200
n_random_trials = 5

v_scr_K_iter = []

net2 = pysmile.Network()
net2.read_file(f"models/DM_screening_rel_point_cond_mut_info_linear.xdsl")
net2.clear_all_evidence()

df_test_w_util_lim = pd.read_csv("models/df_test_new_w_lim.csv", index_col=0)

reader = XMLBIFReader("models/model_bn.xml")
model = reader.get_model()

best_options = get_all_combinations_id_w_optimal_scr(net2, df_test_w_util_lim, limit = limit)
combinations_bn = get_all_combinations_bn(model)

In [ ]:
v_x_K_arr = np.zeros((n_K_points,))
v_x_K = np.zeros((2,))

for ind_k, k in enumerate(np.linspace(0, upper_K, n_K_points)):

    v_K = np.zeros((2,))

    for i, patient_chars in enumerate(best_options.iloc[:,:7].to_dict(orient="records")):
        
        age = patient_chars["Age"]

        infer = VariableElimination(model)
        result = infer.query(variables=list(model.get_parents("CRC")), joint=True)

        patient_chars["Age"] = patient_chars["Age"].replace("age_", "")
        patient_chars["Smoking"] = patient_chars["Smoking"].replace("sm_", "")
        

        evidence = patient_chars
        evidence["Hyperchol."] = patient_chars.pop("Hyperchol_")

        # transform bool values in text
        for key, value in evidence.items():
            if value == 1:
                evidence[key] = "True"
            elif value == 0:
                evidence[key] = "False"

        p_evidence = result.get_value(**evidence)

        p_crc = float(infer.query(variables=["CRC"], evidence=evidence).values[1])
        p_no_crc = float(infer.query(variables=["CRC"], evidence=evidence).values[0])

        try:
            scr = best_options.loc[i, "best_option"]
        except:
            scr = best_options.loc[i, "best_option_w_lim"]
        
        scr_decision_patient = np.unique(["No_screening", scr]).tolist()

        # import pdb; pdb.set_trace()
        
        if scr != "No_screening":
            count_arr = np.zeros(len(scr_decision_patient))
            for _ in range(total_sim):

                # Calculate the expected utility of the citizen for each screening decision
                arr = np.array( [
                        sensitivity(scr) * prob_crc_cit(age) * utility_cit(age, crc=1, r_scr=1, scr = scr, K=k) +
                        (1 - specificity(scr)) * (1-prob_crc_cit(age)) * utility_cit(age, crc=0, r_scr=1, scr = scr, K=k)
                    +
                        (1 - sensitivity(scr)) * prob_crc_cit(age) * utility_cit(age, crc=1, r_scr=0, scr = scr, K=k) +
                        specificity(scr) * (1-prob_crc_cit(age)) * utility_cit(age, crc=0, r_scr=0, scr = scr, K=k)
                    for scr in scr_decision_patient] )

                # Save the decision with highest expected utility.
                argmax = np.argmax(arr)
                count_arr[argmax] += 1
                

            # Approximate the probability of each decision
            p_scr_K = count_arr / total_sim
        else:
            # p_scr_K = [1]
            continue


        # Calculate the expected utility of the government for each incentive amount K
        v_x_K = [ p_scr_K[i] * (
            sensitivity(scr) * p_crc * utility_PM(age, crc=1, r_scr=1, scr = scr, K=k) +
            (1 - specificity(scr)) * p_no_crc * utility_PM(age, crc=0, r_scr=1, scr = scr, K=k)
        +  
            (1 - sensitivity(scr)) * p_crc * utility_PM(age, crc=1, r_scr=0, scr = scr, K=k) +
            specificity(scr) * p_no_crc * utility_PM(age, crc=0, r_scr=0, scr = scr, K=k)
        ) for i, scr in enumerate(scr_decision_patient)] 


        # pdb.set_trace()
        v_K += p_evidence * np.array(v_x_K)

    v_x_K_arr[ind_k] = sum(v_K)

    if ind_k == 5:
        break
    

In [ ]:
v_x_K_arr

array([20.8857369 , 20.98731822, 25.12385899, 28.20068656, 26.80707588,
       29.69585534,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ])

### OBP Simulation

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pgmpy.inference import VariableElimination
from pgmpy.readwrite import XMLBIFReader
import random

from pgmpy.readwrite import XMLBIFReader
import itertools

import pysmile
import pysmile_license
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import pdb
import os

from v2.costs_and_utilities import *
from v2.patients import patient
# from v2.dist_prob_cit import plot_histograms_count_distrib
from v2.get_combinations import *

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

In [ ]:
def generate_obp_grid(z1_start, z1_stop, z1_step, 
                      z2_start, z2_stop, z2_step, 
                      z3_start, z3_stop, z3_step):
    """
    Generates a grid of possible Z = (z1, z2, z3) values for the obp(z) scheme.
    Enforces the constraint: z1 > z2.
    """
    
    # Generate ranges (adding the step to the stop value to make it inclusive)
    z1_values = np.arange(z1_start, z1_stop + z1_step, z1_step)
    z2_values = np.arange(z2_start, z2_stop + z2_step, z2_step)
    z3_values = np.arange(z3_start, z3_stop + z3_step, z3_step)

    valid_combinations = []
    
    # Iterate through all possible combinations
    for z1 in z1_values:
        for z2 in z2_values:
            # Check the primary constraint: 100% threshold must be greater than 50% threshold
            if z1 > z2:
                for z3 in z3_values:
                    valid_combinations.append({
                        "z1_Threshold_100_BP": round(z1, 2),
                        "z2_Threshold_50_BP": round(z2, 2),
                        "z3_Bonus_Euros": round(z3, 2)
                    })
                    
    # Convert the list of dictionaries into a Pandas DataFrame
    df_grid = pd.DataFrame(valid_combinations)
    
    return df_grid


def run_iteration(i, J_SP, J_cit, upper_K, n_K_points, Z_grid, model, best_options):

    v_x_K_arr = np.zeros((n_K_points,))
    expected_util = np.zeros((n_K_points, len(Z_grid)))

    for ind_z, z in enumerate(Z_grid.to_dict(orient="records")):

        for ind_k, k in enumerate(np.linspace(0, upper_K, n_K_points)):

            u_sampled_sp = np.zeros((J_SP,))
            for ind_j in range(J_SP):

                v_K = np.zeros((2,))
                total_screened = 0
                total_detected = 0
                total_cost = 0
            
                for i, patient_chars in enumerate(best_options.iloc[:,:7].to_dict(orient="records")):
                    
                    age = patient_chars["Age"]

                    infer = VariableElimination(model)
                    result = infer.query(variables=list(model.get_parents("CRC")), joint=True)

                    patient_chars["Age"] = patient_chars["Age"].replace("age_", "")
                    patient_chars["Smoking"] = patient_chars["Smoking"].replace("sm_", "")
                    

                    evidence = patient_chars
                    evidence["Hyperchol."] = patient_chars.pop("Hyperchol_")

                    # transform bool values in text
                    for key, value in evidence.items():
                        if value == 1:
                            evidence[key] = "True"
                        elif value == 0:
                            evidence[key] = "False"

                    p_evidence = result.get_value(**evidence)


                    # ---- p_{PM}(c | x) ---- Calculate the probabiltiy of having CRC 
                    p_crc = float(infer.query(variables=["CRC"], evidence=evidence).values[1])
                    p_no_crc = float(infer.query(variables=["CRC"], evidence=evidence).values[0])
                    # -------------------------

                    # ---- Check which is the screening decision given the decision model (Model 2) for the patient profile x
                    try:
                        scr = best_options.loc[i, "best_option"]
                    except:
                        scr = best_options.loc[i, "best_option_w_lim"]
                    scr_decision_patient = np.unique(["No_screening", scr]).tolist()
                    # -------------------------
                    
                    # ----- p_{SP}(s | I, x) ---- Calculate the probability for the citizen to accept screening given incentive K and covariates x
                    # ----- This is done via simulation based on adversarial risk analysis.
                    if scr != "No_screening":
                        count_arr = np.zeros(len(scr_decision_patient))

                        # Simulate utility function for the citizen
                        s_opt = np.zeros((J_cit,))
                        for j_cit in range(J_cit):

                                c_sim = np.random.choice([0,1], p=[1- prob_crc_cit(age), prob_crc_cit(age)])
                                if c_sim == 1 and scr != "No_screening":
                                    r_sim = np.random.choice([0,1], p=[1-sensitivity(scr), sensitivity(scr)])
                                elif c_sim == 0 and scr != "No_screening":
                                    r_sim = np.random.choice([0,1], p=[specificity(scr), 1-specificity(scr)])
                                else:
                                    r_sim = 0

                                total_cost_cit += cost_cit(age, crc=c_sim, scr=scr, r_scr=r_sim, K=k)

                                u_sampled_cit = random_utilities_cit(total_cost_cit)
                                s_opt[j_cit] = np.argmax(u_sampled_cit(total_cost_cit))
        
                        # Approximate the probability of each decision
                        p_scr_K = s_opt.value_counts() / J_cit
                    else:
                        p_scr_K = [1]
                    # ------------------------


                    # ----- Simulate whether the patient is ill, goes to screening and whether cancer is detected.
                    c_sim = np.random.choice([0,1], p=[p_no_crc, p_crc])
                    s_sim = np.random.choice(scr_decision_patient, p=p_scr_K)
                    if c_sim == 1 and s_sim != "No_screening":
                        r_sim = np.random.choice([0,1], p=[1-sensitivity(s_sim), sensitivity(s_sim)])
                    elif c_sim == 0 and s_sim != "No_screening":
                        r_sim = np.random.choice([0,1], p=[specificity(s_sim), 1-specificity(s_sim)])
                    else:
                        r_sim = 0
                    # ------------------------

                    # ----- Calculate cost C(x, c, s, r, k)
                    total_screened += (s_sim != "No_screening")
                    total_detected += (c_sim == 1 and r_sim == 1)
                    total_cost += cost_PM(age, crc=c_sim, scr=s_sim, r_scr=r_sim, K=k)
                    # ------------------------

                    v_K += p_evidence * np.array(total_cost)

                if total_screened / len(best_options) > z[0]:
                    v_K +=  - total_cost
                elif total_screened / len(best_options) > z[1]:
                    v_k += - 0.5 * total_cost

                v_K += z[2] * total_detected

                u_sampled_sp[ind_j] = random_utilities_SP(v_K)


            expected_util[ind_k, ind_z] = np.mean(u_sampled_sp)
       
    return v_x_K_arr

In [12]:
limit = False
J_cit = 10
J_SP = 10

# Define grid of incentives K to evaluate
n_K_points = 20
upper_K = 200

# Define possible Z's (parameterized OBP schemes)
Z_grid = generate_obp_grid(z1_start=0.5, z1_stop=0.7, z1_step=0.1,
                        z2_start=0.2, z2_stop=0.3, z2_step=0.1,
                        z3_start=0, z3_stop=100, z3_step=50)

n_random_trials = 5

v_scr_K_iter = []

net2 = pysmile.Network()
net2.read_file(f"models/DM_screening_rel_point_cond_mut_info_linear.xdsl")
net2.clear_all_evidence()

df_test_w_util_lim = pd.read_csv("models/df_test_new_w_lim.csv", index_col=0)

reader = XMLBIFReader("models/model_bn.xml")
model = reader.get_model()

best_options = get_all_combinations_id_w_optimal_scr(net2, df_test_w_util_lim, limit = limit)
combinations_bn = get_all_combinations_bn(model)

In [13]:
i = 0
run_iteration(i, J_cit, J_SP, upper_K, n_K_points, Z_grid, model, best_options)

> c:\users\danie\appdata\local\temp\ipykernel_25920\3872605280.py(41)run_iteration()

> c:\users\danie\appdata\local\temp\ipykernel_25920\3872605280.py(43)run_iteration()

> c:\users\danie\appdata\local\temp\ipykernel_25920\3872605280.py(45)run_iteration()

> c:\users\danie\appdata\local\temp\ipykernel_25920\3872605280.py(46)run_iteration()



ValueError: probabilities do not sum to 1